In [ ]:

# ============================================================
# CELL 1: BRONZE INGESTION + DAILY RESET + CONTEXT PARAMETERS
# ============================================================

import sys
import json
import uuid
from datetime import datetime
from zoneinfo import ZoneInfo

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, lit, parse_json, current_timestamp

session = get_active_session()

# ------------------------------------------------------------
# 1. CONTEXT PARAMETERS
# These values can be overridden by the notebook schedule.
# ------------------------------------------------------------

params = {}

for arg in sys.argv[1:]:
    if "=" in arg:
        key, value = arg.split("=", 1)
        params[key.upper()] = value

TIMEZONE = params.get("TIMEZONE", "Europe/Amsterdam")
RUN_MODE = params.get("RUN_MODE", "HOURLY")
RESET_ON_NEW_DATE = params.get("RESET_ON_NEW_DATE", "TRUE").upper() == "TRUE"
SOURCE_NAME = params.get("SOURCE_NAME", "SIMULATED_CUSTOMER_API")
ENVIRONMENT = params.get("ENVIRONMENT", "TRIAL")

run_date = datetime.now(ZoneInfo(TIMEZONE)).date()
batch_id = str(uuid.uuid4())

DATABASE = "API_DATA_ENGINEERING"

BRONZE_TABLE = f"{DATABASE}.BRONZE.RAW_API_CUSTOMERS"
SILVER_TABLE = f"{DATABASE}.SILVER.CUSTOMERS"
REJECTS_TABLE = f"{DATABASE}.SILVER.CUSTOMER_REJECTS"
GOLD_OVERVIEW = f"{DATABASE}.GOLD.CUSTOMER_OVERVIEW"
GOLD_COUNTRY = f"{DATABASE}.GOLD.CUSTOMER_COUNTRY_SUMMARY"
GOLD_STATUS = f"{DATABASE}.GOLD.CUSTOMER_STATUS_SUMMARY"

CONTROL_TABLE = f"{DATABASE}.BRONZE.PIPELINE_CONTROL"

print("PIPELINE CONTEXT")
print("Run date:", run_date)
print("Timezone:", TIMEZONE)
print("Run mode:", RUN_MODE)
print("Environment:", ENVIRONMENT)
print("Batch ID:", batch_id)

# ------------------------------------------------------------
# 2. CREATE PIPELINE CONTROL TABLE
# Tracks the last date the pipeline processed.
# ------------------------------------------------------------

session.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
        PIPELINE_NAME VARCHAR(100),
        LAST_RUN_DATE DATE,
        UPDATED_AT TIMESTAMP_NTZ
    )
""").collect()

control_rows = (
    session.table(CONTROL_TABLE)
    .filter(col("PIPELINE_NAME") == lit("CUSTOMER_API_HOURLY_PIPELINE"))
    .select("LAST_RUN_DATE")
    .collect()
)

previous_run_date = (
    control_rows[0]["LAST_RUN_DATE"]
    if control_rows
    else None
)

# ------------------------------------------------------------
# 3. RESET ALL LAYERS ONLY WHEN THE DATE CHANGES
# ------------------------------------------------------------

is_new_date = previous_run_date != run_date

if RESET_ON_NEW_DATE and is_new_date:

    print("NEW DATE DETECTED - RESETTING PIPELINE TABLES")

    tables_to_clear = [
        BRONZE_TABLE,
        SILVER_TABLE,
        REJECTS_TABLE,
        GOLD_OVERVIEW,
        GOLD_COUNTRY,
        GOLD_STATUS
    ]

    for table_name in tables_to_clear:
        session.sql(f"TRUNCATE TABLE IF EXISTS {table_name}").collect()
        print("Cleared:", table_name)

else:
    print("SAME DATE - EXISTING DAILY DATA WILL BE RETAINED")

# Update the control record for this pipeline.
session.sql(f"""
    DELETE FROM {CONTROL_TABLE}
    WHERE PIPELINE_NAME = 'CUSTOMER_API_HOURLY_PIPELINE'
""").collect()

control_df = session.create_dataframe(
    [("CUSTOMER_API_HOURLY_PIPELINE", run_date)],
    schema=["PIPELINE_NAME", "LAST_RUN_DATE"]
)

control_df = control_df.with_column("UPDATED_AT", current_timestamp())

control_df.write.mode("append").save_as_table(
    CONTROL_TABLE,
    column_order="name"
)

# ------------------------------------------------------------
# 4. SIMULATED API CUSTOMER DATA
# Replace this section with real API ingestion when external
# access is available.
# ------------------------------------------------------------

api_response = [
    {
        "customer_id": 1001,
        "customer_name": "John Smith",
        "email": "john.smith@example.com",
        "country": "Netherlands",
        "status": "ACTIVE",
        "city": "Amsterdam",
        "postal_code": "1012"
    },
    {
        "customer_id": 1002,
        "customer_name": "Sarah Johnson",
        "email": "sarah@example.com",
        "country": "Germany",
        "status": "ACTIVE",
        "city": "Berlin",
        "postal_code": "10115"
    },
    {
        "customer_id": 1003,
        "customer_name": "David Brown",
        "email": "david@example.com",
        "country": "France",
        "status": "INACTIVE",
        "city": "Paris",
        "postal_code": "75001"
    },
    {
        "customer_id": 1004,
        "customer_name": "Emma Wilson",
        "email": "emma@example.com",
        "country": "Netherlands",
        "status": "ACTIVE",
        "city": "Rotterdam",
        "postal_code": "3011"
    },
    {
        "customer_id": 1005,
        "customer_name": "Michael Taylor",
        "email": "michael@example.com",
        "country": "Belgium",
        "status": "ACTIVE",
        "city": "Brussels",
        "postal_code": "1000"
    }
]

# ------------------------------------------------------------
# 5. WRITE RAW RECORDS INTO BRONZE
# ------------------------------------------------------------

rows = [
    (batch_id, SOURCE_NAME, json.dumps(customer))
    for customer in api_response
]

source_df = session.create_dataframe(
    rows,
    schema=["BATCH_ID", "SOURCE_NAME", "RAW_JSON"]
)

bronze_df = source_df.select(
    col("BATCH_ID"),
    col("SOURCE_NAME"),
    parse_json(col("RAW_JSON")).alias("RAW_PAYLOAD"),
    current_timestamp().alias("INGESTED_AT")
)

bronze_df.write.mode("append").save_as_table(
    BRONZE_TABLE,
    column_order="name"
)

print("BRONZE INGESTION COMPLETED")
print("Records loaded:", len(api_response))
print("Batch ID:", batch_id)
print("Bronze total records:", session.table(BRONZE_TABLE).count())


In [ ]:

# ============================================================
# CELL 2: SILVER TRANSFORMATION + DATA QUALITY
# ============================================================

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, lit, trim, initcap, lower, upper,
    regexp_like, when, current_timestamp, row_number
)
from snowflake.snowpark.window import Window

session = get_active_session()

DATABASE = "API_DATA_ENGINEERING"

BRONZE_TABLE = f"{DATABASE}.BRONZE.RAW_API_CUSTOMERS"
SILVER_TABLE = f"{DATABASE}.SILVER.CUSTOMERS"
REJECTS_TABLE = f"{DATABASE}.SILVER.CUSTOMER_REJECTS"

# ------------------------------------------------------------
# 1. READ BRONZE DATA
# ------------------------------------------------------------

bronze_df = session.table(BRONZE_TABLE)

# ------------------------------------------------------------
# 2. EXTRACT AND STANDARDIZE JSON FIELDS
# ------------------------------------------------------------

silver_source = bronze_df.select(
    col("RAW_PAYLOAD")["customer_id"].cast("NUMBER(38,0)").alias("CUSTOMER_ID"),
    initcap(trim(col("RAW_PAYLOAD")["customer_name"].cast("STRING"))).alias("CUSTOMER_NAME"),
    lower(trim(col("RAW_PAYLOAD")["email"].cast("STRING"))).alias("EMAIL"),
    initcap(trim(col("RAW_PAYLOAD")["country"].cast("STRING"))).alias("COUNTRY"),
    upper(trim(col("RAW_PAYLOAD")["status"].cast("STRING"))).alias("STATUS"),
    initcap(trim(col("RAW_PAYLOAD")["city"].cast("STRING"))).alias("CITY"),
    trim(col("RAW_PAYLOAD")["postal_code"].cast("STRING")).alias("POSTAL_CODE"),
    col("BATCH_ID"),
    col("INGESTED_AT")
)

# ------------------------------------------------------------
# 3. DEDUPLICATE CUSTOMER RECORDS
# Keep the latest record for each customer ID.
# ------------------------------------------------------------

dedup_window = (
    Window.partition_by("CUSTOMER_ID")
    .order_by(
        col("INGESTED_AT").desc(),
        col("BATCH_ID").desc()
    )
)

deduplicated_df = (
    silver_source
    .with_column("ROW_NUM", row_number().over(dedup_window))
    .filter(col("ROW_NUM") == 1)
    .drop("ROW_NUM")
)

# ------------------------------------------------------------
# 4. DATA QUALITY VALIDATION
# ------------------------------------------------------------

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

validated_df = deduplicated_df.with_column(
    "REJECTION_REASON",
    when(
        col("CUSTOMER_ID").is_null() | (col("CUSTOMER_ID") <= 0),
        lit("INVALID CUSTOMER ID")
    )
    .when(
        col("CUSTOMER_NAME").is_null() | (col("CUSTOMER_NAME") == ""),
        lit("MISSING CUSTOMER NAME")
    )
    .when(
        col("EMAIL").is_null() |
        (~regexp_like(col("EMAIL"), lit(email_pattern))),
        lit("INVALID EMAIL")
    )
    .when(
        col("COUNTRY").is_null() | (col("COUNTRY") == ""),
        lit("MISSING COUNTRY")
    )
    .when(
        ~col("STATUS").isin(["ACTIVE", "INACTIVE"]),
        lit("INVALID CUSTOMER STATUS")
    )
    .otherwise(lit(None))
)

# ------------------------------------------------------------
# 5. SPLIT VALID AND REJECTED RECORDS
# ------------------------------------------------------------

valid_df = validated_df.filter(
    col("REJECTION_REASON").is_null()
)

rejected_df = validated_df.filter(
    col("REJECTION_REASON").is_not_null()
)

# ------------------------------------------------------------
# 6. PREPARE SILVER CUSTOMER TABLE
# ------------------------------------------------------------

silver_output = (
    valid_df.select(
        "CUSTOMER_ID",
        "CUSTOMER_NAME",
        "EMAIL",
        "COUNTRY",
        "STATUS",
        "CITY",
        "POSTAL_CODE",
        "BATCH_ID",
        "INGESTED_AT"
    )
    .with_column("PROCESSED_AT", current_timestamp())
)

# Refresh Silver on each run to prevent hourly duplicates.
silver_output.write.mode("overwrite").save_as_table(
    SILVER_TABLE,
    column_order="name"
)

# ------------------------------------------------------------
# 7. PREPARE REJECTS TABLE
# ------------------------------------------------------------

rejects_output = rejected_df.select(
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "EMAIL",
    "BATCH_ID",
    "REJECTION_REASON",
    "INGESTED_AT"
).with_column(
    "REJECTED_AT",
    current_timestamp()
)

rejects_output.write.mode("overwrite").save_as_table(
    REJECTS_TABLE,
    column_order="name"
)

# ------------------------------------------------------------
# 8. QUALITY RESULTS
# ------------------------------------------------------------

print("SILVER TRANSFORMATION COMPLETED")
print("--------------------------------")
print("Bronze records:", bronze_df.count())
print("Unique customer records:", deduplicated_df.count())
print("Valid customers:", session.table(SILVER_TABLE).count())
print("Rejected customers:", session.table(REJECTS_TABLE).count())

print("\nSILVER CUSTOMER DATA:")
session.table(SILVER_TABLE).show()

print("\nREJECTED CUSTOMER DATA:")
session.table(REJECTS_TABLE).show()


In [ ]:

# ============================================================
# CELL 3: GOLD REPORTING AND AGGREGATIONS
# ============================================================

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, count, sum as sum_, when, lit,
    current_timestamp, round as round_
)

session = get_active_session()

DATABASE = "API_DATA_ENGINEERING"

SILVER_TABLE = f"{DATABASE}.SILVER.CUSTOMERS"

GOLD_OVERVIEW = f"{DATABASE}.GOLD.CUSTOMER_OVERVIEW"
GOLD_COUNTRY = f"{DATABASE}.GOLD.CUSTOMER_COUNTRY_SUMMARY"
GOLD_STATUS = f"{DATABASE}.GOLD.CUSTOMER_STATUS_SUMMARY"

# ------------------------------------------------------------
# 1. READ SILVER DATA
# ------------------------------------------------------------

silver_df = session.table(SILVER_TABLE)

total_customers = silver_df.count()

print("GOLD PROCESSING STARTED")
print("Total Silver customers:", total_customers)

# ------------------------------------------------------------
# 2. CUSTOMER OVERVIEW
# Detailed customer-level reporting table
# ------------------------------------------------------------

customer_overview = (
    silver_df.select(
        "CUSTOMER_ID",
        "CUSTOMER_NAME",
        "EMAIL",
        "COUNTRY",
        "STATUS",
        "CITY",
        "POSTAL_CODE",
        "BATCH_ID",
        "INGESTED_AT",
        "PROCESSED_AT"
    )
    .with_column("GOLD_CREATED_AT", current_timestamp())
)

# ------------------------------------------------------------
# 3. COUNTRY SUMMARY
# Customers grouped by country
# ------------------------------------------------------------

country_summary = (
    silver_df.group_by("COUNTRY")
    .agg(
        count(lit(1)).alias("TOTAL_CUSTOMERS"),
        sum_(
            when(col("STATUS") == "ACTIVE", 1).otherwise(0)
        ).alias("ACTIVE_CUSTOMERS"),
        sum_(
            when(col("STATUS") == "INACTIVE", 1).otherwise(0)
        ).alias("INACTIVE_CUSTOMERS")
    )
    .with_column(
        "ACTIVE_PERCENTAGE",
        round_(
            col("ACTIVE_CUSTOMERS") * 100 /
            col("TOTAL_CUSTOMERS"), 2
        )
    )
    .with_column("GOLD_CREATED_AT", current_timestamp())
)

# ------------------------------------------------------------
# 4. STATUS SUMMARY
# Customers grouped by status
# ------------------------------------------------------------

status_summary = (
    silver_df.group_by("STATUS")
    .agg(
        count(lit(1)).alias("TOTAL_CUSTOMERS"),
        count(col("COUNTRY")).alias("CUSTOMERS_WITH_COUNTRY"),
        count(col("CITY")).alias("CUSTOMERS_WITH_CITY")
    )
)

if total_customers > 0:
    status_summary = status_summary.with_column(
        "CUSTOMER_PERCENTAGE",
        round_(
            col("TOTAL_CUSTOMERS") * 100 /
            lit(total_customers), 2
        )
    )
else:
    status_summary = status_summary.with_column(
        "CUSTOMER_PERCENTAGE", lit(0)
    )

status_summary = status_summary.with_column(
    "GOLD_CREATED_AT", current_timestamp()
)

# ------------------------------------------------------------
# 5. REFRESH GOLD TABLES
# Overwrite ensures hourly runs do not accumulate duplicates.
# ------------------------------------------------------------

customer_overview.write.mode("overwrite").save_as_table(
    GOLD_OVERVIEW,
    column_order="name"
)

country_summary.write.mode("overwrite").save_as_table(
    GOLD_COUNTRY,
    column_order="name"
)

status_summary.write.mode("overwrite").save_as_table(
    GOLD_STATUS,
    column_order="name"
)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nGOLD PROCESSING COMPLETED")
print("--------------------------------")
print("Customer overview:", session.table(GOLD_OVERVIEW).count())
print("Country summary:", session.table(GOLD_COUNTRY).count())
print("Status summary:", session.table(GOLD_STATUS).count())

print("\nCUSTOMER OVERVIEW:")
session.table(GOLD_OVERVIEW).show()

print("\nCOUNTRY SUMMARY:")
session.table(GOLD_COUNTRY).show()

print("\nSTATUS SUMMARY:")
session.table(GOLD_STATUS).show()
